# 데이터별 전처리 — 단계별 실제 표

DEG용 데이터(GSE142025·GSE30529)를 예로, 전처리 각 단계를 **실제 데이터 전/후**로 확인.
- 1. probe→gene (중복 병합) · 2. log2 · 3. 정규화 · 4. 라벨 · 5. 최종 결과

In [1]:
import pandas as pd, numpy as np, gzip, collections
from io import StringIO
pd.set_option("display.max_columns",10)
B="../01_preprocessing/bulk/output"; PROC="../../data/processed"
mp=pd.read_csv("probe_gene_map.csv")   # 실제 probe→gene (hgu133a2)
def series_all(g):
    with gzip.open(f"{B}/02_normalized/{g}_series_matrix.txt.gz","rt") as f:
        rows=[];on=False
        for l in f:
            if l.startswith('"ID_REF"'): on=True; rows.append(l); continue
            if on:
                if l.startswith("!"): break
                rows.append(l)
    return pd.read_csv(StringIO("".join(rows)),sep="\t",index_col=0)
print("준비 완료 | 매핑",mp.shape[0],"개")

준비 완료 | 매핑 22529 개


## 1. probe → gene 매핑 + 중복 병합 (GSE30529, 실제 hgu133a2 매핑)

**① probe + gene 결합** — probe는 다른데(1405_i_at·204655_at) 유전자는 같음(CCL5) = 중복

In [2]:
s=series_all("GSE30529")
j=s.join(mp.set_index("PROBEID")["SYMBOL"]).reset_index().rename(columns={"ID_REF":"probe","SYMBOL":"gene"})
view=j[["probe","gene"]+list(s.columns[:3])]
pd.concat([view.head(4), view[view.gene=="CCL5"]]).reset_index(drop=True)

,probe,gene,GSM757014,GSM757015,GSM757016
0,1007_s_at,DDR1,0.155938,0.095946,0.401841
1,1053_at,RFC2,0.236349,0.119733,0.081872
2,117_at,HSPA6,-0.155251,0.173947,-0.033306
3,121_at,PAX8,0.090568,0.440096,0.225213
4,1405_i_at,CCL5,-0.226266,1.299415,1.109525
5,204655_at,CCL5,-0.104441,1.360925,1.035378


**② 중복은 평균(avereps)** → CCL5 두 probe가 1행으로

In [3]:
pr=[p for p in mp[mp.SYMBOL=="CCL5"].PROBEID if p in s.index]
m=pd.DataFrame({"probes":[", ".join(pr),"1007_s_at","1053_at","117_at"],"n_probe":[len(pr),1,1,1]},
               index=["CCL5","DDR1","RFC2","HSPA6"])
for c in s.columns[:3]:
    m[c]=[round(s.loc[pr,c].mean(),4),round(s.loc["1007_s_at",c],4),round(s.loc["1053_at",c],4),round(s.loc["117_at",c],4)]
m.index.name="gene"; m

,probes,n_probe,GSM757014,GSM757015,GSM757016
gene,,,,,
CCL5,"1405_i_at, 204655_at",2,-0.1654,1.3302,1.0725
DDR1,1007_s_at,1,0.1559,0.0959,0.4018
RFC2,1053_at,1,0.2363,0.1197,0.0819
HSPA6,117_at,1,-0.1553,0.1739,-0.0333


**③ 전체 중복 유전자 (probe 2개 이상)**

In [4]:
cnt=mp.groupby("SYMBOL")["PROBEID"].agg(n_probe="count",probes=lambda x:", ".join(x))
dup=cnt[cnt.n_probe>=2].sort_values("n_probe",ascending=False)
print(f"전체 {cnt.shape[0]}개 유전자 중 probe 2개 이상 = {dup.shape[0]}개 (평균으로 합쳐짐)")
dup.head(6)

전체 13631개 유전자 중 probe 2개 이상 = 5188개 (평균으로 합쳐짐)


,n_probe,probes
SYMBOL,,
IGHG1,21,"211430_s_at, 211633_x_at, 211636_at, 211647_x_at, 211648_at, 211868_x_at, 214916_x_at, 215721_at, 216510_x_at, 216541_x_at, 216542_x_at, 216557_x_at, 216558_x_at, 216892_at, 217169_at, 217198_x_at, 217222_at, 217236_x_at, 217260_x_at, 217360_x_at, 217369_at"
IGHM,16,"209374_s_at, 211430_s_at, 211634_x_at, 211636_at, 211868_x_at, 212827_at, 214916_x_at, 215949_x_at, 216491_x_at, 216510_x_at, 216541_x_at, 216542_x_at, 216557_x_at, 216558_x_at, 217169_at, 217360_x_at"
HFE,13,"206086_x_at, 206087_x_at, 210864_x_at, 211326_x_at, 211327_x_at, 211328_x_at, 211329_x_at, 211330_s_at, 211331_x_at, 211332_x_at, 211863_x_at, 211866_x_at, 214647_s_at"
CSHL1,11,"202493_x_at, 205958_x_at, 206475_x_at, 207285_x_at, 208068_x_at, 208069_x_at, 208293_x_at, 208294_x_at, 208295_x_at, 208356_x_at, 211739_x_at"
IGHA1,11,"211636_at, 211868_x_at, 214916_x_at, 216510_x_at, 216541_x_at, 216542_x_at, 216557_x_at, 216558_x_at, 217022_s_at, 217169_at, 217360_x_at"
CFLAR,11,"208485_x_at, 209508_x_at, 209939_x_at, 210563_x_at, 210564_x_at, 211316_x_at, 211317_s_at, 211862_x_at, 214486_x_at, 214618_at, 217654_at"


## 2. log2 — 큰 값 압축 (GSE30529, 값 큰 유전자)
현재 값은 log2 상태 → **2^값 = 원래 선형값**으로 되돌려 비교.

In [5]:
g=pd.read_csv(f"{B}/01_genematrix/GSE30529.genematrix.txt",sep="\t",index_col=0)
big=g.mean(1).sort_values(ascending=False).head(5).index; sm=list(g.columns[:3])
print("[log2 변환 전 — 원래 선형값, 수천~수만]")
(2**g.loc[big,sm]).round(0)

[log2 변환 전 — 원래 선형값, 수천~수만]


,GSM757014,GSM757015,GSM757016
geneNames,,,
RPL41,24367.0,26003.0,24531.0
UBB,24283.0,22381.0,21947.0
EEF1A1,18117.0,17664.0,23743.0
RPS3A,16998.0,17986.0,19208.0
RPS29,14527.0,19709.0,13584.0


In [6]:
print("[log2 변환 후 — log2, 좁게 압축]")
g.loc[big,sm].round(2)

[log2 변환 후 — log2, 좁게 압축]


,GSM757014,GSM757015,GSM757016
geneNames,,,
RPL41,14.57,14.67,14.58
UBB,14.57,14.45,14.42
EEF1A1,14.15,14.11,14.54
RPS3A,14.05,14.13,14.23
RPS29,13.83,14.27,13.73


## 3. 정규화 — 샘플별 밝기 편차 통일 (여러 데이터셋 병합본, 실제)
개별 데이터는 이미 정규화돼 오지만, ML 병합본엔 정규화 전(preNorm) 스냅샷이 있어 효과가 보임. (라벨 제거해 표시)

In [7]:
pre=pd.read_csv(f"{PROC}/data.train.preNorm.txt",sep="\t",index_col=0)
post=pd.read_csv(f"{PROC}/data.train.txt",sep="\t",index_col=0)
pre.columns=[c.rsplit("_",1)[0] for c in pre.columns]; post.columns=[c.rsplit("_",1)[0] for c in post.columns]
order=pre.mean().sort_values(); sm=[order.index[0],order.index[-1],order.index[1]]  # 어두움·밝음(가운데)·어두움
print("[정규화 전] 밝은 샘플(가운데)이 값 통째로 큼")
pre[sm].head(5).round(2)

[정규화 전] 밝은 샘플(가운데)이 값 통째로 큼


,GSE96804_GSM2544294,GSE104948_GSM2810771,GSE96804_GSM2544304
geneNames,,,
ISG15,7.01,9.66,6.95
AGRN,6.69,10.08,6.80
SCNN1D,5.61,4.43,5.36
CPTP,5.18,8.66,5.31
VWA1,5.88,6.53,5.95


In [8]:
print("[정규화 후] 밝은 샘플이 내려와 서로 맞춰짐")
post[sm].head(5).round(2)

[정규화 후] 밝은 샘플이 내려와 서로 맞춰짐


,GSE96804_GSM2544294,GSE104948_GSM2810771,GSE96804_GSM2544304
geneNames,,,
ISG15,7.60,8.24,7.48
AGRN,7.37,8.10,7.63
SCNN1D,5.36,5.05,5.00
CPTP,5.53,6.77,5.93
VWA1,5.81,6.14,6.00


In [9]:
m=pd.DataFrame({"정규화 전 평균":pre.mean().round(3),"정규화 후 평균":post.mean().round(3)})
ex=m.sort_values("정규화 전 평균")
print("샘플 평균 — 전 낮은3 + 높은3 (전엔 5.8~7.9로 벌어짐 → 후엔 6점대로 모임)")
pd.concat([ex.head(3),ex.tail(3)])

샘플 평균 — 전 낮은3 + 높은3 (전엔 5.8~7.9로 벌어짐 → 후엔 6점대로 모임)


,정규화 전 평균,정규화 후 평균
GSE96804_GSM2544294,5.771,6.254
GSE96804_GSM2544304,5.787,6.298
GSE96804_GSM2544290,5.793,6.326
GSE104948_GSM2810793,7.866,6.668
GSE104948_GSM2810773,7.898,6.691
GSE104948_GSM2810771,7.924,6.706


## 4. 그룹 라벨 — Control/Early/Late (GSE142025, 각 2개)

In [10]:
n=pd.read_csv(f"{B}/02_normalized/GSE142025.normalized.txt",sep="\t",index_col=0)
pick2=lambda lab:[c for c in n.columns if c.endswith(lab)][:2]
cols=pick2("Control")+pick2("Early")+pick2("Late")
after=n[cols].head(5); before=after.copy(); before.columns=[c.rsplit("_",1)[0] for c in after.columns]
print("[라벨 전] 열 = 원래 이름 (N/B/A)")
before.round(2)

[라벨 전] 열 = 원래 이름 (N/B/A)


,N2A,N3A,B3B,B4B,A11A,A12A
geneNames,,,,,,
A1BG,6.47,6.20,6.39,6.33,6.87,6.95
A1BG-AS1,5.72,5.46,5.47,6.17,5.57,6.30
A1CF,12.19,12.51,11.96,12.84,10.92,11.27
A2M,15.48,15.00,15.23,15.46,16.26,16.36
A2M-AS1,7.45,7.51,7.56,7.75,8.01,7.35


In [11]:
print("[라벨 후] 열 = 이름_그룹 (N→Control, B→Early, A→Late)")
after.round(2)

[라벨 후] 열 = 이름_그룹 (N→Control, B→Early, A→Late)


,N2A_Control,N3A_Control,B3B_Early,B4B_Early,A11A_Late,A12A_Late
geneNames,,,,,,
A1BG,6.47,6.20,6.39,6.33,6.87,6.95
A1BG-AS1,5.72,5.46,5.47,6.17,5.57,6.30
A1CF,12.19,12.51,11.96,12.84,10.92,11.27
A2M,15.48,15.00,15.23,15.46,16.26,16.36
A2M-AS1,7.45,7.51,7.56,7.75,8.01,7.35


## 5. ★ 최종 전처리 결과 (GSE142025) — DEG에 바로 넣는 통합 발현행렬

In [12]:
print("행(유전자) × 열(샘플):",n.shape)
print("라벨 종류별:",dict(collections.Counter(c.rsplit("_",1)[1] for c in n.columns)))
n[cols].head(6).round(2)

행(유전자) × 열(샘플): (17184, 36)
라벨 종류별: {'Late': 21, 'Early': 6, 'Control': 9}


,N2A_Control,N3A_Control,B3B_Early,B4B_Early,A11A_Late,A12A_Late
geneNames,,,,,,
A1BG,6.47,6.20,6.39,6.33,6.87,6.95
A1BG-AS1,5.72,5.46,5.47,6.17,5.57,6.30
A1CF,12.19,12.51,11.96,12.84,10.92,11.27
A2M,15.48,15.00,15.23,15.46,16.26,16.36
A2M-AS1,7.45,7.51,7.56,7.75,8.01,7.35
A4GALT,11.16,11.05,11.53,10.98,11.66,11.67
